# Actividad 5: Transfer Learning con un modelo elegido por ustedes

**Curso:** Deep Learning  
**Profesor:** Gonzalo A. Ruz  
**Ayudante:** Anthony D. Cho

En esta actividad deberán aplicar **transfer learning** a un problema de clasificación de imágenes.

Usaremos el dataset **cats vs dogs**, pero esta vez deberán elegir un modelo preentrenado desde `keras.applications` distinto a los vistos en clase.

## Objetivo
Comparar dos etapas:

1. **Feature extraction**: usar la base congelada.
2. **Fine-tuning**: descongelar parte de la base y continuar el entrenamiento.

## Modelos permitidos
Elijan **uno** de los siguientes:

- `DenseNet121`
- `Xception`
- `InceptionV3`
- `VGG16`
- `VGG19`

## Restricción
No usar:

- `MobileNetV2`
- `ResNet50`
- `EfficientNetB0`

## Instrucciones
- La actividad debe ser realizada por los grupos de trabajo
- Responda cada pregunta en las celdas correspondientes
- Justifique brevemente sus respuestas cuando se solicite
- Renombrar el archivo agregando el apellido de las y los integrantes, por ejemplo Actividad5_Tupper_Tudor_Gorosito_Acosta.ipynb
- Subir el archivo al link de entrega Actividad 5 en webcursos que será habilitado
- __Fecha de entrega:__ Idealmente al final del bloque 2 de la clase del 20 de abril 2026. Fecha límite de entrega 27 de abril 2026

## Integrantes (RUT - Nombre y apellido):

-

-

-

## Instrucciones generales

1. Elijan un modelo de la lista.
2. Carguen la base preentrenada correctamente.
3. Construyan un modelo para **feature extraction**.
4. Entrénenlo y registren el resultado.
5. Luego hagan un **fine-tuning corto**.
6. Comparen ambos resultados.
7. Respondan las preguntas finales.

## Sugerencia práctica
Mantengan el notebook simple y no cambien el dataset.
La novedad aquí debe estar en el **modelo elegido** y en el flujo de transfer learning.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

## Dataset

Usaremos el mismo dataset de la sesión:

- `train`
- `validation`
- `test`

Asegúrense de que la ruta base corresponda a su Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Ajusta esta ruta según tu Google Drive
base_dir = "/content/drive/MyDrive/data/cats-vs-dogs_small"

train_dir = os.path.join(base_dir, "train")
validation_dir = os.path.join(base_dir, "validation")
test_dir = os.path.join(base_dir, "test")

print("Train dir exists:", os.path.exists(train_dir))
print("Validation dir exists:", os.path.exists(validation_dir))
print("Test dir exists:", os.path.exists(test_dir))

In [4]:
img_size = (160, 160)
batch_size = 32
seed = 123

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    label_mode="binary",
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    seed=seed
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    label_mode="binary",
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    label_mode="binary",
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False
)

class_names = train_ds.class_names
print("Class names:", class_names)

In [6]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## Data augmentation

Usaremos una estrategia simple de augmentation para entrenamiento.

In [7]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ],
    name="data_augmentation"
)

## Pregunta 1: elegir el modelo

Completen la celda de abajo con el modelo que eligieron y con su función de preprocesamiento correspondiente.

### Recordatorio
Cada familia de modelos puede requerir un `preprocess_input` distinto.

In [ ]:
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input

model_name = "DenseNet121"
base_model = DenseNet121(
    input_shape=img_size + (3,),
    include_top=False,
    weights="imagenet"
)


## Verificación básica

Comprueben que el modelo base fue cargado correctamente.

In [ ]:
print("Modelo elegido:", model_name)
print("Base model cargado:", base_model is not None)

## Pregunta 2: feature extraction

En esta primera etapa deben:

- congelar la base,
- construir una cabeza de clasificación nueva,
- compilar el modelo,
- y entrenarlo por pocas épocas.

In [ ]:
base_model.trainable = False

inputs = keras.Input(shape=img_size + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

initial_epochs = 6
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=initial_epochs
)


## Resultado de feature extraction

Registren aquí el desempeño obtenido al final de esta etapa.

In [ ]:
val_loss_fe, val_acc_fe = model.evaluate(val_ds, verbose=0)
test_loss_fe, test_acc_fe = model.evaluate(test_ds, verbose=0)

print("Feature extraction - val accuracy:", val_acc_fe)
print("Feature extraction - test accuracy:", test_acc_fe)

## Pregunta 3: fine-tuning

Ahora hagan un ajuste fino del modelo.

Sugerencia:

- descongelen la base,
- congelen una parte inicial,
- dejen entrenables solo las capas finales,
- recompilen con learning rate pequeño.

In [ ]:
base_model.trainable = True

fine_tune_at = int(len(base_model.layers) * 0.7)
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_epochs = 4
total_epochs = initial_epochs + fine_tune_epochs

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1
)


## Resultado de fine-tuning

In [ ]:
val_loss_ft, val_acc_ft = model.evaluate(val_ds, verbose=0)
test_loss_ft, test_acc_ft = model.evaluate(test_ds, verbose=0)

print("Fine-tuning - val accuracy:", val_acc_ft)
print("Fine-tuning - test accuracy:", test_acc_ft)

## Curvas de entrenamiento

Grafiquen las curvas combinadas de accuracy y loss.

In [ ]:
acc_key = "accuracy" if "accuracy" in history.history else "acc"
val_acc_key = "val_accuracy" if "val_accuracy" in history.history else "val_acc"

acc = history.history[acc_key] + history_fine.history[acc_key]
val_acc = history.history[val_acc_key] + history_fine.history[val_acc_key]

loss = history.history["loss"] + history_fine.history["loss"]
val_loss = history.history["val_loss"] + history_fine.history["val_loss"]

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, label="Train Accuracy")
plt.plot(val_acc, label="Val Accuracy")
plt.axvline(x=initial_epochs - 1, linestyle="--", label="Start Fine-Tuning")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(loss, label="Train Loss")
plt.plot(val_loss, label="Val Loss")
plt.axvline(x=initial_epochs - 1, linestyle="--", label="Start Fine-Tuning")
plt.legend()
plt.title("Loss")

plt.show()


## Pregunta 4: Visualizar algunas predicciones

Muestren 4 imágenes del conjunto de test con:

- etiqueta real,
- predicción del modelo.

In [ ]:
batch_images, batch_labels = next(iter(test_ds.take(1)))
pred_probs = model.predict(batch_images[:4], verbose=0).flatten()
pred_labels = (pred_probs >= 0.5).astype(int)

plt.figure(figsize=(12, 3))
for i in range(4):
    ax = plt.subplot(1, 4, i + 1)
    plt.imshow(batch_images[i].numpy().astype("uint8"))
    real = class_names[int(batch_labels[i].numpy())]
    pred = class_names[int(pred_labels[i])]
    prob = pred_probs[i]
    plt.title(f"Real: {real}\nPred: {pred}\nProb: {prob:.3f}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Pregunta 5: Tabla resumen

Completen esta tabla con sus resultados.

| Modelo elegido | Val acc feature extraction | Test acc feature extraction | Val acc fine-tuning | Test acc fine-tuning |
|---|---:|---:|---:|---:|
| DenseNet121 | `val_acc_fe` | `test_acc_fe` | `val_acc_ft` | `test_acc_ft` |


## Pregunta 6: Preguntas de reflexión

Respondan brevemente:

1. ¿Qué modelo eligieron y por qué?
2. ¿Qué diferencia observaron entre feature extraction y fine-tuning?
3. ¿Qué punto de corte usaron para el fine-tuning y cómo lo decidieron?
4. ¿Creen que su modelo fue una buena elección para este problema? Justifiquen brevemente.

1. **Modelo elegido:** DenseNet121 con pesos ImageNet; buen equilibrio entre capacidad y costo computacional para cats vs dogs.
2. **Feature extraction vs fine-tuning:** comparar `val_acc_fe/test_acc_fe` con `val_acc_ft/test_acc_ft`; en general el fine-tuning mejora al adaptar capas finales al dominio.
3. **Punto de corte:** se congeló el 70% inicial (`fine_tune_at = int(len(base_model.layers) * 0.7)`) y se entrenó el 30% final para ajustar representaciones altas sin sobreajustar.
4. **Evaluación de la elección:** si `val_acc_ft` y `test_acc_ft` superan los valores de feature extraction con brecha train/val estable, DenseNet121 fue una buena elección para este problema.


## Suerte!